# L02 · Actor-Critic / A2C

对应 lecture: `lectures/02-actor-critic.md`

**目标**：
- 直接调用 `a2c_minimal.train()` 跑 50k step CartPole
- 与 sb3 A2C 对比
- 观察 actor / critic / entropy loss 三条曲线


In [ ]:
import sys, os, types
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import gymnasium as gym

## 1. 手写 A2C — 调用 a2c_minimal.train()

In [ ]:
from a2c_minimal import train

args = types.SimpleNamespace(
    env='CartPole-v1', n_envs=4, total_steps=20_000,
    n_steps=5, hidden=64, lr=7e-4, gamma=0.99,
    vf_coef=0.5, ent_coef=0.01, max_grad_norm=0.5,
    seed=42, log_interval=50, cpu=True,
)
train(args)

## 2. sb3 A2C 对照

In [ ]:
from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

train_env = DummyVecEnv([lambda: gym.make('CartPole-v1') for _ in range(4)])
train_env = VecMonitor(train_env)

model = A2C('MlpPolicy', train_env, learning_rate=7e-4, n_steps=5, gamma=0.99,
            vf_coef=0.5, ent_coef=0.0, seed=42, verbose=0)
model.learn(total_timesteps=20_000)

eval_env = gym.make('CartPole-v1')
mean_R, std_R = evaluate_policy(model, eval_env, n_eval_episodes=20)
print(f'sb3 A2C eval: {mean_R:.1f} ± {std_R:.1f}')

## 3. 自我检查

- [ ] 手写 A2C 在 20k step 达到 ep mean ≥ 100
- [ ] sb3 A2C 在同样 budget 下达到 ≥ 200
- [ ] 理解 1-step TD advantage 公式
